# Newton's Law of Cooling — First-Order Linear SVR (Quantum Kernel)

Solves Newton's law of cooling with the support vector regression (SVR) method and a quantum kernel.

$$\dot T = r\,(T_{\text{env}} - T(t)),\qquad T(0)=T_0,$$
analytic solution $T(t) = T_{\text{env}} + (T_0 - T_{\text{env}})\,e^{-rt}$.

This is a **first-order linear** ODE, so it uses exactly the same SVR machinery as our linear example — only the equation definition changes.

## Mapping to the solver's form

The solver expects $T' + p(t)\,T + s(t) = 0$. Expanding Newton's law:
$$\dot T = r T_{\text{env}} - r T \;\;\Longrightarrow\;\; T' + \underbrace{r}_{p}\,T + \underbrace{(-r T_{\text{env}})}_{s} = 0.$$
So the coefficient of $T$ is the constant cooling rate $p=r$, and the forcing term is the constant $s=-rT_{\text{env}}$. Since the equation is linear, no dummy variables are needed and the SVR reduces to a single linear solve.

**Demo scenario:** a hot drink at $T_0=90^\circ$C cooling toward room temperature $T_{\text{env}}=25^\circ$C, with rate $r=3$ over $t\in[0,1]$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import Statevector

In [ ]:
# ============================================================
# QUANTUM KERNEL (same architecture, cached statevectors)
# ============================================================
def build_HEA(n_qubits=8, depth=1, seed=1):
    rng = np.random.default_rng(seed)
    theta = rng.uniform(0, 2*np.pi, size=(depth, n_qubits, 2))
    qc = QuantumCircuit(n_qubits, name="HEA")
    for i in range(depth):
        for q in range(n_qubits):
            qc.rx(theta[i, q, 0], q); qc.rz(theta[i, q, 1], q)
        for q in range(n_qubits-1):
            qc.cx(q, q+1)
    return qc

def build_kernel_feature_map(n_qubits=8):
    x = Parameter("x")
    qc = QuantumCircuit(n_qubits, name="KernelMap")
    qc.compose(build_HEA(n_qubits, 1, seed=1), inplace=True)
    for q in range(n_qubits): qc.rx((q*x)/2, q)
    qc.compose(build_HEA(n_qubits, 1, seed=2), inplace=True)
    for q in range(n_qubits): qc.rx((q*x)/2, q)
    return qc

N_QUBITS = 8
_QC = build_kernel_feature_map(N_QUBITS)
_PX = [p for p in _QC.parameters if p.name == "x"][0]
_psi_cache = {}
def _psi(t):
    key = round(float(t), 12)
    if key not in _psi_cache:
        _psi_cache[key] = Statevector.from_instruction(_QC.assign_parameters({_PX: t})).data
    return _psi_cache[key]
def kappa(u, v):
    return np.abs(np.vdot(_psi(v), _psi(u)))**2

print("Kernel ready. kappa(0.3,0.3) =", round(kappa(0.3, 0.3), 6))

In [ ]:
# ============================================================
# KERNEL DERIVATIVES  K[q,p](u,v) = d^q/du^q d^p/dv^p kappa
# ============================================================
DERIV_H = 1e-3
def K00(u, v): return kappa(u, v)
def K10(u, v): return (kappa(u+DERIV_H, v) - kappa(u-DERIV_H, v)) / (2*DERIV_H)
def K01(u, v): return (kappa(u, v+DERIV_H) - kappa(u, v-DERIV_H)) / (2*DERIV_H)
def K11(u, v):
    return (kappa(u+DERIV_H, v+DERIV_H) - kappa(u+DERIV_H, v-DERIV_H)
            - kappa(u-DERIV_H, v+DERIV_H) + kappa(u-DERIV_H, v-DERIV_H)) / (4*DERIV_H**2)
print("Derivative helpers ready.")

In [ ]:
# ============================================================
# NEWTON'S LAW OF COOLING:  T' + r T - r T_env = 0
# ============================================================
R_COOL = 3.0      # cooling rate r  (coefficient of T)
T_ENV  = 25.0     # ambient temperature
T0     = 90.0     # initial temperature T(0)
t0     = 0.0      # initial time

P_COEF   = R_COOL              # coefficient of T  (constant)
FORCING  = -R_COOL * T_ENV     # forcing term s(t) (constant)

def T_exact(t):
    return T_ENV + (T0 - T_ENV) * np.exp(-R_COOL * t)

print(f"ODE: dT/dt = {R_COOL} ({T_ENV} - T),  T(0) = {T0}")
print(f"Analytic: T(t) = {T_ENV} + ({T0-T_ENV}) exp(-{R_COOL} t)")

In [ ]:
# ============================================================
# LINEAR SVR SOLVER  (constant coefficient p, constant forcing s)
# ============================================================
class CoolingSVRSolver:
    def __init__(self, t_train, gamma=1e5):
        self.X = np.asarray(t_train, float)
        self.N = len(self.X)
        self.gamma = gamma

    def fit(self, p_const, forcing_const, t0, T0):
        X, N = self.X, self.N
        p = p_const * np.ones(N)
        g = self.gamma

        M11 = np.array([[K11(X[j], X[i]) for j in range(N)] for i in range(N)])
        M10 = np.array([[K10(X[j], X[i]) for j in range(N)] for i in range(N)])
        M01 = np.array([[K01(X[j], X[i]) for j in range(N)] for i in range(N)])
        M00 = np.array([[K00(X[j], X[i]) for j in range(N)] for i in range(N)])
        b01 = np.array([K01(t0, X[i]) for i in range(N)])
        b00 = np.array([K00(t0, X[i]) for i in range(N)])
        c10 = np.array([K10(X[j], t0) for j in range(N)])
        c00 = np.array([K00(X[j], t0) for j in range(N)])
        s00 = K00(t0, t0)

        A = np.zeros((N+1, N+1)); rhs = np.zeros(N+1)
        for i in range(N):
            for j in range(N):
                core = M11[i,j] + p[i]*M10[i,j] + p[j]*M01[i,j] + p[i]*p[j]*M00[i,j]
                if i == j: core += 1.0/g
                A[i, j] = core - p[j]*(b01[i] + p[i]*b00[i])
            A[i, N] = p[i]
            rhs[i]  = -forcing_const          # constant forcing
        for j in range(N):
            A[N, j] = c10[j] + p[j]*c00[j] - p[j]*s00
        A[N, N] = 1.0
        rhs[N]  = T0

        sol = np.linalg.solve(A, rhs)
        self.alpha = sol[:N]; self.b = sol[N]
        self.beta  = -np.sum(self.alpha * p)
        self.p, self.t0 = p, t0
        print(f"Solved {N+1}x{N+1} linear system | cond(A) = {np.linalg.cond(A):.2e}")
        return self

    def predict(self, t_eval):
        X, N, p = self.X, self.N, self.p
        out = []
        for t in np.atleast_1d(np.asarray(t_eval, float)):
            out.append(sum(self.alpha[j]*(K10(X[j], t) + p[j]*K00(X[j], t)) for j in range(N))
                       + self.beta*K00(self.t0, t) + self.b)
        return np.array(out)

print("Cooling SVR solver defined.")

In [ ]:
# ============================================================
# SOLVE
# ============================================================
t_train = np.linspace(0, 1, 21)
t_test  = np.linspace(0, 1, 200)

solver = CoolingSVRSolver(t_train, gamma=1e5)
solver.fit(P_COEF, FORCING, t0, T0)

T_pred = solver.predict(t_test)
T_ref  = T_exact(t_test)
err = np.abs(T_pred - T_ref); rng = T_ref.max() - T_ref.min()
print(f"\nT_pred(0) = {T_pred[0]:.4f}  (target {T0})")
print(f"max abs error      = {err.max():.4e}")
print(f"max normalized err = {err.max()/rng:.4e}")

In [ ]:
# ============================================================
# PLOTS
# ============================================================
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(t_test, T_ref, "b-", lw=2.5, label="Analytic solution")
ax[0].plot(t_test, T_pred, "r--", lw=2, label="SVR + Quantum Kernel")
ax[0].scatter(t_train, T_exact(t_train), color="green", s=45, zorder=5,
              edgecolors="darkgreen", label="Training points")
ax[0].axhline(T_ENV, color="gray", ls=":", lw=1.5, label=f"T_env = {T_ENV}")
ax[0].set_xlabel("time t"); ax[0].set_ylabel("Temperature T(t)")
ax[0].set_title("Newton Cooling: SVR Quantum Kernel vs Analytic", fontweight="bold")
ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].semilogy(t_test, err / rng, "g-", lw=2)
ax[1].set_xlabel("time t"); ax[1].set_ylabel("normalized error")
ax[1].set_title("Normalized Error (log scale)", fontweight="bold")
ax[1].grid(alpha=0.3, which="both")

plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# SUMMARY
# ============================================================
print("="*60)
print("NEWTON'S LAW OF COOLING  -  FIRST-ORDER SVR")
print("="*60)
print(f"Equation : dT/dt = r (T_env - T),  r={R_COOL}, T_env={T_ENV}")
print(f"IC       : T(0) = {T0}")
print(f"Kernel   : 2x(HEA depth-1 + feature map), {N_QUBITS} qubits")
print(f"Training : {len(t_train)} points, gamma = {solver.gamma:.0e}")
print("-"*60)
print(f"T_pred(0)            = {T_pred[0]:.4f}  (target {T0})")
print(f"max absolute error   = {err.max():.4e}")
print(f"max normalized error = {err.max()/rng:.4e}")
print("="*60)